# 03: HMM/HSMM Analysis

This notebook fits Hidden Markov Models to identify behavioral states.

**METHODOLOGICAL CORRECTIONS:**
1. **Input:** Uses standardized features (not raw activity) to reduce noise
2. **State Selection:** Based on BIOLOGICAL INTERPRETABILITY, not AIC/BIC
   - AIC/BIC decrease monotonically for long time series and are UNRELIABLE
   - Select K=3 or K=4 based on mean activity separation and diel patterns
3. **Dwell-Time Filtering:** Enforces minimum dwell time (≥4 samples = 1 hour)
   - Fixes "state flickering" problem caused by noisy observations
4. **Biological Validation:** States labeled by mean activity (Rest, Low, Active, Highly Active)

**Correct Pipeline:** Raw activity → Features → Standardization → HMM → Dwell-time filter → Validated states → Cosinor

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from loguru import logger

from src import io, models_hmm

# Fix matplotlib overflow error with large datasets
plt.rcParams['agg.path.chunksize'] = 10000

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style('whitegrid')

## 1. Load Feature Data

In [ ]:
df = io.load_raw_data(Path('../data/processed/feature_data.parquet'))
logger.info(f"Loaded data with shape: {df.shape}")

# Check for standardized features
if 'activity_log_standardized' in df.columns:
    logger.success("Found standardized features - ready for HMM")
else:
    logger.warning("No standardized features found - consider re-running 02_features.ipynb")

df.head()

## 2. Prepare Features for HMM

Use standardized features as HMM input. Raw activity mixes behavioral states and is inappropriate.

In [ ]:
# Prepare feature matrix for HMM
# Use standardized log-transformed activity
if 'activity_log_standardized' in df.columns:
    feature_col = 'activity_log_standardized'
elif 'activity_standardized' in df.columns:
    feature_col = 'activity_standardized'
else:
    logger.warning("No standardized features - using raw activity (NOT RECOMMENDED)")
    feature_col = 'activity'

# Clean data and prepare
df_clean = df.dropna(subset=[feature_col])
X = df_clean[feature_col].values.reshape(-1, 1)
activity_raw = df_clean['activity'].values  # Keep raw for interpretation

logger.info(f"Using feature: {feature_col}")
logger.info(f"HMM input shape: {X.shape}")
logger.info(f"Feature mean: {X.mean():.3f}, std: {X.std():.3f}")

## 3. Biological State Selection

**IMPORTANT:** We do NOT use AIC/BIC for state selection because:
- Long time series cause monotonic decrease in IC
- IC-based selection is unreliable and not biologically meaningful

Instead, we evaluate K=2..4 based on:
1. Separation of mean activity levels
2. Dwell-time distributions (biologically plausible durations)
3. Diel (day/night) prevalence patterns

In [ ]:
# Biological state selection
timestamps = df_clean['timestamp'] if 'timestamp' in df_clean.columns else None

bio_results = models_hmm.select_states_biologically(
    X=X,
    activity=activity_raw,
    state_range=range(2, 5),  # Test K=2, 3, 4
    timestamps=timestamps,
    random_state=RANDOM_STATE
)

# Display characteristics for each K
print("\n" + "="*80)
print("BIOLOGICAL STATE CHARACTERISTICS")
print("="*80)
for k, chars in bio_results['characteristics'].items():
    print(f"\nK={k} states:")
    print(chars.to_string())
    print(f"\nMean activity range: {chars['mean_activity'].min():.1f} - {chars['mean_activity'].max():.1f}")
    print(f"Mean dwell time range: {chars['mean_dwell_time'].min():.1f} - {chars['mean_dwell_time'].max():.1f} samples")

## 4. Select Final K and Fit Model

Based on biological validation above, select K (typically 3 or 4).

In [ ]:
# Select K based on inspection of characteristics above
# Recommended: K=3 (Rest, Low Activity, Active) or K=4 (Rest, Low, Active, Highly Active)
SELECTED_K = 3  # CHANGE THIS based on biological validation above

logger.info(f"\n{'='*80}")
logger.info(f"FITTING FINAL MODEL with K={SELECTED_K} states")
logger.info(f"Justification: Based on biological interpretability, not IC")
logger.info(f"{'='*80}\n")

# Fit final model
final_model = models_hmm.fit_gaussian_hmm(
    X,
    n_states=SELECTED_K,
    n_iter=100,
    random_state=RANDOM_STATE
)

# Predict states
states_raw, posteriors = models_hmm.predict_states(final_model, X)

# Apply minimum dwell-time filter (CRITICAL for fixing flickering)
states_filtered = models_hmm.apply_minimum_dwell_time(states_raw, min_dwell=4)

logger.success(f"States predicted and dwell-time filtered")

# Analyze final state characteristics
state_chars = models_hmm.analyze_state_characteristics(
    states_filtered,
    activity_raw,
    timestamps
)

print("\nFINAL STATE CHARACTERISTICS:")
print(state_chars.to_string())

## 5. Assign Biological Labels

Label states based on mean activity levels for interpretability.

In [ ]:
# Assign biological labels
bio_labels = models_hmm.assign_biological_labels(state_chars, n_states=SELECTED_K)

print("\nBIOLOGICAL STATE LABELS:")
for state_id, label in bio_labels.items():
    mean_act = state_chars[state_chars['state']==state_id]['mean_activity'].values[0]
    print(f"  State {state_id} → '{label}' (mean activity: {mean_act:.1f})")

# Add labeled column to results
state_labels_map = bio_labels
df_clean['hmm_state_raw'] = states_raw
df_clean['hmm_state'] = states_filtered
df_clean['state_label'] = df_clean['hmm_state'].map(state_labels_map)

logger.success("Biological labels assigned")

## 6. Visualize State Sequences

In [ ]:
# Plot state sequences with data
plot_len = min(500, len(states_filtered))
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Original data
axes[0].plot(X[:plot_len], label='Data')
axes[0].set_ylabel('Value')
axes[0].set_title('Original Time Series')
axes[0].legend()
axes[0].grid(True)

# Hidden states
axes[1].plot(states_filtered[:plot_len], drawstyle='steps-post')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Hidden State')
axes[1].set_title('Predicted Hidden States (Dwell-Time Filtered)')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('../outputs/hmm_states.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Transition Matrix

In [ ]:
# Display transition matrix
trans_matrix = final_model.transmat_
print("\nTransition Matrix:")
print(trans_matrix)

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(trans_matrix, annot=True, fmt='.3f', cmap='Blues')
plt.title('State Transition Matrix')
plt.xlabel('To State')
plt.ylabel('From State')
plt.tight_layout()
plt.savefig('../outputs/hmm_transition_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Save Results

In [ ]:
# Save state sequences and posteriors
df_clean['hmm_state_raw'] = states_raw
df_clean['hmm_state'] = states_filtered  # Use filtered states
df_clean['state_label'] = df_clean['hmm_state'].map(bio_labels)

# Add posterior probabilities
for i in range(posteriors.shape[1]):
    df_clean[f'state_{i}_prob'] = posteriors[:, i]

output_path = Path('../data/processed/hmm_data.parquet')
io.save_processed_data(df_clean, output_path, format='parquet')
logger.success(f"HMM results saved to {output_path}")

In [ ]:
params = {
    'random_state': RANDOM_STATE,
    'selected_k': SELECTED_K,
    'selection_method': 'biological_validation',
    'dwell_time_filtered': True,
    'min_dwell_samples': 4,
    'n_samples': len(X),
    'feature_used': feature_col,
    'biological_labels': bio_labels,
}
logger.info(f"HMM parameters: {params}")